# Learning Embeddings — Smokers/Friends/Cancer with a Structural Axis

This notebook revisits the smokers-friends-cancer embedding example from
[`7-learning_embeddings.ipynb`](./7-learning_embeddings.ipynb), replacing its
constant-based grounding with a **structural dimension**. It illustrates how
the same knowledge base can be expressed more compactly by quantifying over a
structural axis rather than enumerating named constants one by one.

The previous notebook represents the 14 people as named constants `A` through
`N`, each bound to its own learnable embedding tensor, with facts written as
closed formulas over those constants, for example:

$$
Friends(A, B), \quad Smokes(A), \quad Cancer(E).
$$

This notebook instead introduces a structural axis `p` ranging over the
14-person domain. A single learnable embedding matrix represents all people
at once, and formulas quantify over structural positions rather than over
individually declared constants. The predicate networks, embedding size
(`embedding_size = 15`), and overall knowledge base are otherwise unchanged
from the constant-based version.

## Main differences from the constant-based version

| Constant-based version | Structural-axis version |
|---|---|
| 14 constants `A` through `N` | one structural dimension `p` with `N = 14` |
| one embedding tensor per person | one `nn.Embedding(N, embedding_size)` matrix |
| facts written as closed formulas over constants | facts represented as tensors, read through structural guards and masks |
| `Friends(A, B)` | guarded formula over `p` and its alias `p1` |
| `Smokes(A)` | masked unary supervision over `p` |
| `Cancer(E)` | masked unary supervision over `p` |

## Fact representation

Ground-truth facts are represented as tensors rather than as individual
closed formulas:

- `f_pos[p, p1]`: an `N × N` binary matrix marking known positive friendship pairs;
- `f_neg[p, p1]`: an `N × N` binary matrix marking known negative friendship pairs;
- `y_s[p]`: smoking labels;
- `m_s(p)`: structural guard marking which smoking labels are observed;
- `y_c[p]`: cancer labels;
- `m_c(p)`: structural guard marking which cancer labels are observed.

The friendship matrices are used directly as structural guards in quantified
formulas, while the unary smoking and cancer supervision combine a structural
mask with an equivalence formula.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Ground-truth data

This section builds the observed facts for the 14-person domain as plain
tensors, which the signature and formulas defined later will read from.

The 14 people are split into two groups, exactly as in the constant-based notebook:

- group 1: `A` through `H`;
- group 2: `I` through `N`.

Friendship is represented by two `N × N` binary matrices:

- `f_pos`: `1` marks a known friend pair;
- `f_neg`: `1` marks a known non-friend pair within the same group.

Smoking and cancer are represented by label vectors together with observation masks:

- `y_s`: smoking labels;
- `m_s`: smoking observation mask (all 14 labels are known);
- `y_c`: cancer labels;
- `m_c`: cancer observation mask (`1` only for people in group 1).

As in the constant-based notebook, complete cancer supervision is available
for group 1 only.

In [ ]:
g  = set('ABCDEFGHIJKLMN')
g1 = set('ABCDEFGH');  g2 = set('IJKLMN')
persons  = sorted(g)
N        = len(persons)           # 14
idx      = {p: i for i, p in enumerate(persons)}

friends_pairs = [('A','B'),('A','E'),('A','F'),('A','G'),
                 ('B','C'),('C','D'),('E','F'),('G','H'),
                 ('I','J'),('J','M'),('K','L'),('M','N')]
smokes_set = set('AEFGJN')
cancer_set = set('AE')

# ── Binary friendship matrices ────────────────────────────────────────────
f_pos = torch.zeros(N, N)   # 1 = known friend
f_neg = torch.zeros(N, N)   # 1 = known non-friend (within group, undirected)

for u, v in friends_pairs:
    f_pos[idx[u], idx[v]] = f_pos[idx[v], idx[u]] = 1.0

for grp in [sorted(g1), sorted(g2)]:
    for u in grp:
        for v in grp:
            if u != v and f_pos[idx[u], idx[v]] == 0.0:
                f_neg[idx[u], idx[v]] = 1.0

# ── Unary supervision tensors  (N, 1) ─────────────────────────────────
y_s = torch.tensor([[1.0 if p in smokes_set else 0.0] for p in persons])
m_s = torch.ones(N)   # all smokes labels known

y_c = torch.tensor([[1.0 if p in cancer_set else 0.0] for p in persons])
m_c = torch.tensor([1.0 if p in g1         else 0.0 for p in persons])

print(f"Persons: {N}  |  Friends+ edges: {int(f_pos.sum())}  |  Friends- edges: {int(f_neg.sum())}")
print(f"Smokers: {smokes_set}  |  Cancer: {cancer_set}")


## 2. Signature

The signature declares the symbolic vocabulary, together with the structural
dimension that replaces the 14 named constants.

The dimension `p` ranges over the 14 person slots. Formulas that need two
independent person positions (e.g. friendship) use the alias `p1` for the
second occurrence of the same dimension.

The signature contains:

- the sort `Person`;
- the structural relations `friends_pos` and `friends_neg` (fixed friendship guards), and `m_s`, `m_c` (supervision masks);
- tensor-valued constants for labels (`y_s`, `y_c`) and the embedding matrix (`x`);
- neural predicates `Smokes`, `Cancer`, and `Friends`.

The constant `x : Person[p]` is the full `(N, embedding_size)` embedding
matrix, indexed by the structural dimension `p`.

In [ ]:
import sltn
from sltn import Signature, Interpretation, KB
from sltn import Logic
from sltn.interpretation.operators import aggregator, conjunction, disjunction, implication, negation

embedding_size = 15
EPS, stable = 1e-12, True

sig    = Signature("smokers_struct")
sig.sort("Person")

p_dim  = sig.dimension("p")   # person structural dimension (14 slots)

# ── Structural fact tables ────────────────────────────────────────────────
friends_pos_pred = sig.structural_relation("friends_pos", ["p", "p"])  # (N, N)
friends_neg_pred = sig.structural_relation("friends_neg", ["p", "p"])  # (N, N)
sig.structural_relation("m_s", ["p"])  # (N, 1)
sig.structural_relation("m_c", ["p"])  # (N, 1)

# ── Variables ─────────────────────────────────────────────────────────────
sig.constant("y_s", "Bool",   dims=["p"])   # smokes labels      (N, 1)
sig.constant("y_c", "Bool",   dims=["p"])   # cancer labels      (N, 1)

# ── Embedding constant ─────────────────────────────────────────────────────
sig.constant("x", "Person", dims=["p"])   # embedding matrix  (N, E)

# ── Neural predicates ─────────────────────────────────────────────────────
sig.predicate("Smokes",      ["Person"], input_dims=["p"], output_dims=["p"])
sig.predicate("Cancer",      ["Person"], input_dims=["p"], output_dims=["p"])
# Binary: output_dims=[p_dim, p_dim] → produces (1, N, N, 1) for all ordered pairs
sig.predicate("Friends",  ["Person","Person"], input_dims=["p", "p"], output_dims=["p", "p"])

print(sig)

## 3. Formulas

This section defines the supervision formulas, rule formulas, and query
formulas exactly as they are parsed in the code below. Every formula
quantifies over the structural dimension `p` (and its alias `p1`) rather
than over individually named constants.

### Supervision formulas

Smoking and cancer supervision use masked equivalence, so that the
aggregation only considers positions with an observed label:

$$
\forall p \mid m_s(p): (Smokes(x[p])[p] \leftrightarrow y_s[p])
$$

$$
\forall p \mid m_c(p): (Cancer(x[p])[p] \leftrightarrow y_c[p])
$$

Friendship supervision uses the fixed structural relations as guards instead of a mask:

$$
\forall p, p1 \mid friends\_pos(p, p1): Friends(x[p], x[p1])[p, p1]
$$

$$
\forall p, p1 \mid friends\_neg(p, p1): \lnot Friends(x[p], x[p1])[p, p1]
$$

### Rule formulas

The rules carry the same meaning as in the constant-based notebook — anti-
reflexive and symmetric friendship, existence of a friend, propagation of
smoking through friendship, and the smoking/cancer implications — but they
now quantify over structural positions instead of the variables `x`, `y`:

$$
\forall p: \lnot Friends(x[p], x[p])[p, p]
$$

$$
\forall p, p1: (Friends(x[p], x[p1])[p, p1] \to Friends(x[p1], x[p])[p1, p])
$$

$$
\forall p: \exists p1: Friends(x[p], x[p1])[p, p1]
$$

$$
\forall p, p1: ((Friends(x[p], x[p1])[p, p1] \land Smokes(x[p])[p]) \to Smokes(x[p1])[p1])
$$

$$
\forall p: (Smokes(x[p])[p] \to Cancer(x[p])[p])
$$

$$
\forall p: (\lnot Smokes(x[p])[p] \to \lnot Cancer(x[p])[p])
$$

In [ ]:
# ── Supervision facts ─────────────────────────────────────────────────────
phi_smokes_pos = sig.parse("∀ p | m_s(p): (Smokes(x[p])[p] <-> y_s[p])")
phi_cancer_sup = sig.parse("∀ p | m_c(p): (Cancer(x[p])[p] <-> y_c[p])")
phi_friends_pos = sig.parse("∀ p, p1 | friends_pos(p, p1): Friends(x[p], x[p1])[p, p1]")
phi_friends_neg = sig.parse("∀ p, p1 | friends_neg(p, p1): ¬Friends(x[p], x[p1])[p, p1]")

# ── Rules ─────────────────────────────────────────────────────────────────
phi_anti_refl        = sig.parse("∀ p: ¬Friends(x[p], x[p])[p, p]")
phi_sym              = sig.parse("∀ p, p1: (Friends(x[p], x[p1])[p, p1] -> Friends(x[p1], x[p])[p1, p])")
phi_has_friend       = sig.parse("∀ p: ∃ p1: Friends(x[p], x[p1])[p, p1]")
phi_spread           = sig.parse("∀ p, p1: ((Friends(x[p], x[p1])[p, p1] & Smokes(x[p])[p]) -> Smokes(x[p1])[p1])")
phi_smoke_cancer     = sig.parse("∀ p: (Smokes(x[p])[p] -> Cancer(x[p])[p])")
phi_nosmoke_nocancer = sig.parse("∀ p: (¬Smokes(x[p])[p] -> ¬Cancer(x[p])[p])")

# ── Knowledge base (10 clauses — same structure as original) ──────────────
phi_kb = KB([
    phi_smokes_pos, phi_cancer_sup, phi_friends_pos, 
    phi_friends_neg, phi_anti_refl, phi_sym, phi_has_friend, 
    phi_spread, phi_smoke_cancer, phi_nosmoke_nocancer,
])

# ── Query formulas (not in KB) ────────────────────────────────────────────
phi_q1 = sig.parse("forall p: (Cancer(x[p])[p] -> Smokes(x[p])[p])")
phi_q2 = sig.parse("∀ p, p1: ((Cancer(x[p])[p] & Cancer(x[p1])[p1]) -> Friends(x[p], x[p1])[p, p1])")

print("phi_kb clauses:", len(phi_kb.clauses))
for phi in phi_kb.clauses:
    print(" ", phi)

## 4. Neural models and embedding matrix

This section defines the predicate networks and replaces the 14 separate
learnable tensors from the constant-based notebook with a single embedding
matrix. The embedding layer `person_emb = nn.Embedding(N, embedding_size)`
has shape `N × embedding_size`, where `N = 14`.

The predicates are grounded by simple MLPs, matching the constant-based notebook:

- `Smokes` maps one person embedding to a truth value;
- `Cancer` maps one person embedding to a truth value;
- `Friends` maps a pair of person embeddings to a truth value.

For `Friends`, the structural-axis machinery supplies aligned embedding
tensors for the two person axes `p` and `p1`. The model concatenates the two
embeddings along the feature dimension and applies the friendship MLP over
all ordered pairs.

In [ ]:
class MLP(nn.Module):
    def __init__(self, layer_sizes):
        super().__init__()
        self.elu     = nn.ELU()
        self.sigmoid = nn.Sigmoid()
        self.layers  = nn.ModuleList([
            nn.Linear(layer_sizes[i-1], layer_sizes[i])
            for i in range(1, len(layer_sizes))
        ])

    def forward(self, x):
        for layer in self.layers[:-1]: x = self.elu(layer(x))
        return self.sigmoid(self.layers[-1](x))
    
class FriendsFn(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, v1, v2):
        return self.net(torch.cat([v1, v2], dim=-1))


smokes_net  = MLP([embedding_size, 16, 16, 1])
cancer_net  = MLP([embedding_size, 16, 16, 1])
friends_net = MLP([embedding_size * 2, 16, 16, 1])
friends_fn = FriendsFn(friends_net)

# Single embedding matrix — replaces 14 individual tensors
person_emb = nn.Embedding(N, embedding_size)

## 5. Interpretation

The interpretation grounds the structural signature in tensors, neural
modules, and fuzzy logic operators.

Key groundings are:

- `Person`: an embedding tensor type of size `embedding_size`;
- `x`: the full learnable embedding matrix;
- `friends_pos`, `friends_neg`: the fixed positive and negative friendship matrices;
- `y_s`, `m_s`, `y_c`, `m_c`: fixed supervision label and mask tensors;
- `Smokes`, `Cancer`, `Friends`: neural predicates.

The structural relations and supervision tensors are fixed for the whole
run. The embedding matrix is the only grounding that is learnable, through
`person_emb.weight`.



In [ ]:
def make_logic(p_exists=2):
    logic = Logic()
    logic["∧"] = conjunction.AndProd(stable=stable, eps=EPS)
    logic["∨"] = disjunction.OrProbSum(stable=stable, eps=EPS)
    logic["¬"] = negation.NotStandard()
    logic["→"] = implication.ImpliesReichenbach(stable=stable, eps=EPS)
    logic["↔"] = implication.EquivSimilarity(p=1.0, eps=EPS)
    logic["∀"] = aggregator.AggregPMeanError(p=2, stable=stable, eps=EPS)
    logic["∃"] = aggregator.AggregPMean(p=p_exists, stable=stable, eps=EPS)
    logic["SatAgg"] = aggregator.AggregPMeanError(p=2, stable=stable, eps=EPS)
    return logic

def build_interpretation(logic) -> Interpretation:
    interp = Interpretation(sig, logic=logic).to(device)

    interp["Person"] = sltn.Type("Person", shape=(embedding_size,))
    interp["Bool"] = sltn.Bool

    interp.logic.StructAgg = {"p": {"forall" : aggregator.AggregPMean(p=1/10, stable=stable, eps=EPS),
                                    "exists" : aggregator.AggregPMean(p=2, stable=stable, eps=EPS)},
                                }
    # Identity predicates for supervision data
    for name in ("SmokesLabel", "SmokesMask", "CancerLabel", "CancerMask"):
        interp[name] = lambda v: v

    # Neural predicates
    # Lambda wrapper: required so forall p, p1 context works correctly
    interp["Smokes"] = smokes_net
    interp["Cancer"] = cancer_net
    interp["Friends"] = friends_fn

    # Structural fact tables (fixed for the whole run)
    interp[friends_pos_pred] = f_pos
    interp[friends_neg_pred] = f_neg

    # Data tensors (rebuilt each step to track embedding gradients — see training loop)
    interp["y_s"] = y_s
    interp["m_s"] = m_s
    interp["y_c"] = y_c
    interp["m_c"] = m_c

    interp["x"] = person_emb.weight

    return interp

logic = make_logic(p_exists=1)   # curriculum: start lenient
interp = build_interpretation(logic)

# Quick sanity check
with torch.no_grad():
    sat0 = interp(phi_kb).value.item()
print(f"Initial KB satisfaction: {sat0:.4f}")

## 6. Training loop

Training optimizes the satisfaction of the KB for 2000 epochs using Adam.

The loop uses **`kb_describe(phi_kb)`** to name each clause and
**`kb_backward(..., aggregator="satagg")`** to aggregate all 10 clauses with
`SatAgg` and backpropagate `1 - SatAgg` in a single call, printing the
per-clause satisfaction every 100 epochs together with the two query
formulas `phi_q1`, `phi_q2` (evaluated separately, since they are not part
of the training KB).

In [ ]:
from sltn import kb_describe, kb_backward

params = (
    list(smokes_net.parameters()) +
    list(cancer_net.parameters()) +
    list(friends_net.parameters()) +
    list(person_emb.parameters())   # ← replaces 14 individual embedding tensors
)
optimizer = torch.optim.Adam(params, lr=0.001)
kb_dict = kb_describe(phi_kb)

for epoch in range(2000):
    # Curriculum: lenient ∃ (p=1) for first 200 epochs, then strict (p=6)
    p = 1 if epoch <= 200 else 1
    interp.logic["∃"].p = p
    interp.logic["∀"].p = p
    interp.logic["SatAgg"].p = p

    optimizer.zero_grad()
    stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
    optimizer.step()

    if epoch % 100 == 0:
        losses_str = " | ".join(
            f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
        )
        w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
        print(f"epoch {epoch:4d} | [{losses_str}] | α=[{w_str}]")

        if epoch % 100 == 0:
            with torch.no_grad():
                phi1_val = interp(phi_q1).value
                phi2_val = interp(phi_q2).value
            print(f"| Phi1 {phi1_val.item():.3f} | Phi2 {phi2_val.item():.3f}")
            

## 7. Visualizing results

The first set of plots shows the observed ground-truth facts:

- `Smokes(p)` and `Cancer(p)` per person;
- known friendship facts within group 1 and group 2;
- white cells indicate unknown or unused entries.

The second set of plots shows the learned predicate values after training:

- `Smokes(x[p])[p]`;
- `Cancer(x[p])[p]`;
- `Friends(x[p], x[p1])[p, p1]`.

The friendship plots show the full predicate evaluated over all ordered pairs within each group.

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt

g1s = sorted(g1)
g2s = sorted(g2)

def plt_heatmap(df, vmin=None, vmax=None):
    plt.pcolor(df, vmin=vmin, vmax=vmax)
    plt.yticks(np.arange(0.5, len(df.index), 1), df.index)
    plt.xticks(np.arange(0.5, len(df.columns), 1), df.columns)
    plt.colorbar()

# ── Ground-truth ──────────────────────────────────────────────────────────
df_facts_sc = pd.DataFrame(
    np.array([[(p in smokes_set), (p in cancer_set) if p in g1 else math.nan]
              for p in persons]),
    columns=["Smokes(p)", "Cancer(p)"], index=persons)

df_facts_f1 = pd.DataFrame(
    np.array([[float((u,v) in friends_pairs or (v,u) in friends_pairs)
               if u != v else math.nan for u in g1s] for v in g1s]),
    index=g1s, columns=g1s)

df_facts_f2 = pd.DataFrame(
    np.array([[float((u,v) in friends_pairs or (v,u) in friends_pairs)
               if u != v else math.nan for u in g2s] for v in g2s]),
    index=g2s, columns=g2s)

plt.figure(figsize=(14, 3))
plt.subplot(131); plt_heatmap(df_facts_sc, 0, 1); plt.title("Facts: Smokes, Cancer")
plt.subplot(132); plt_heatmap(df_facts_f1, 0, 1); plt.title("Friends Group 1")
plt.subplot(133); plt_heatmap(df_facts_f2, 0, 1); plt.title("Friends Group 2")
plt.suptitle("Ground-truth facts  (white = unknown)")
plt.tight_layout()
plt.show()

# ── Learned predicate values ──────────────────────────────────────────────

with torch.no_grad():
    emb_stack = person_emb(torch.arange(N)).to(device)           # (N, E)
    s_vals = smokes_net(emb_stack).squeeze().cpu().numpy()
    c_vals = cancer_net(emb_stack).squeeze().cpu().numpy()
    # Full (N, N) Friends matrix via cross-product
    v1 = emb_stack.unsqueeze(1).expand(N, N, embedding_size)      # (N, N, E)
    v2 = emb_stack.unsqueeze(0).expand(N, N, embedding_size)      # (N, N, E)
    f_vals = friends_net(torch.cat([v1, v2], dim=-1)).squeeze(-1).cpu().numpy()  # (N, N)

df_pred_sc = pd.DataFrame(np.stack([s_vals, c_vals], axis=1),
                           columns=["Smokes(p)", "Cancer(p)"], index=persons)
df_pred_f1 = pd.DataFrame(f_vals[:8, :8], index=g1s, columns=g1s)
df_pred_f2 = pd.DataFrame(f_vals[8:, 8:], index=g2s, columns=g2s)
# for df in [df_pred_f1, df_pred_f2]:
#     df.values[range(len(df)), range(len(df))] = math.nan

plt.figure(figsize=(14, 3))
plt.subplot(131); plt_heatmap(df_pred_sc, 0, 1); plt.title("Learned: Smokes, Cancer")
plt.subplot(132); plt_heatmap(df_pred_f1, 0, 1); plt.title("Learned: Friends Group 1")
plt.subplot(133); plt_heatmap(df_pred_f2, 0, 1); plt.title("Learned: Friends Group 2")
plt.suptitle("Learned predicate values after training")
plt.tight_layout()
plt.show()


## 8. Clause-level satisfaction

This plot shows the final satisfaction value of each of the 10 KB clauses.
It helps identify which supervision formulas and logical rules are well
satisfied and which remain in tension with the rest of the KB.

In [ ]:
with torch.no_grad():
    clause_sats = [interp(clause).value.item() for clause in phi_kb.clauses]

clause_labels = [str(phi_kb.clauses[i])[:80] for i in range(len(phi_kb.clauses))]

plt.figure(figsize=(16, len(phi_kb.clauses) * 0.5 + 1))
plt.barh(clause_labels, clause_sats, color='steelblue')
plt.xlim(0, 1); plt.xlabel("Satisfaction")
plt.title("KB clause satisfaction after training")
plt.tight_layout()
plt.show()